In [6]:
import re
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

#Enter your file path
file_path = "/content/drive/MyDrive/DL/Data.tsv"

#Load the dataset
df = pd.read_csv(file_path, sep="\t")

#Remove column 1 'Entry'
df = df.iloc[:, 1:]

#Rename column names
new_column_names = {
    'Entry Name': 'Name',
    'Subcellular location [CC]': 'Location',
    'Sequence': 'Sequence',
    'Length': 'Length'
}

df = df.rename(columns=new_column_names)

#Modify the data in 'Name' column
df['Name'] = df['Name'].str.replace('_HUMAN', '')  #removing "_HUMAN" from the column

#Filter the data based on sequence length
df = df[df['Sequence'].apply(lambda x: len(x) <= 500)].reset_index(drop=True)

#Remove the texts inside '{}' and '[]' in column 2
#Modify colum 3
def remove_brackets_content(text):
    text = str(text)
    return re.sub(r'\{.*?\}', '', text) # function for removing string inside '{}'

def remove_sqbrackets_content(text):
    text = str(text)
    return re.sub(r'\[.*?\]', '', text) # function for removing every string inside '[]'

df['Location'] = df['Location'].apply(remove_brackets_content)
df['Location'] = df['Location'].apply(remove_sqbrackets_content)

#Specify the strings that we need to keep - "Subcellular locations"
def keep_only_specified_locations(location_string):
    # Convert to string if not already
    location_string = str(location_string)
    allowed_locations = ["Nucleus", "Cytoplasm", "Secreted"]
    found_locations = []
    for loc in allowed_locations:
        if loc in location_string:
            found_locations.append(loc)
    return "; ".join(found_locations) if found_locations else ""  # Function for only returning the allowed locations if present, and return empty cell otherwise


df['Location'] = df['Location'].apply(keep_only_specified_locations)

#Removing the rows with empty location cells
df = df[df['Location'].notna()]
df = df[df['Location'] != '']

#Adding a new column which indicates "1" or "0" for presence/absence of the protein in that location
def add_location_columns(df):
    allowed_locations = ["Nucleus", "Cytoplasm", "Secreted"]
    for location in allowed_locations:
        df[location] = df['Location'].str.contains(location).astype(int)
    return df # Function to create new columns which return 1/0 based on presence/absence of the protein in the given location

df = add_location_columns(df)

#Remove the original location column
df = df.drop('Location', axis=1)

#Specify your output path
output_path = '/content/drive/MyDrive/DL/modified_data.csv'

#Saving the modified csv file
df.to_csv(output_path, index=False) #Writing the modified csv file for downstream analysis
print(len(df.index))
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
6358


,Name,Sequence,Nucleus,Cytoplasm,Secreted
1,MOTSC,MRWQEMGYIFYPRKLR,1,0,1
2,CD3CH,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,0,0,1
3,NBDY,MGDQPCASGRSTLPPGNAREAKPPKKRCLLAPRWDYPEGTPNGGST...,0,1,0
5,MED19,MENFTALFGAQADPPPPPTALGFGPGKPPPPPPPPAGGGPGTAPPP...,1,0,0
6,IGLC7,GQPKAAPSVTLFPPSSEELQANKATLVCLVSDFNPGAVTVAWKADG...,0,0,1
